# Stage 2: Metadata Injection + Container Hash Breaking
This notebook is **STAGE 2** of your workflow.
It takes your final CapCut export and injects 3-layer metadata (iTunes ilst + 3GPP + XMP),
Apple device fingerprinting, GPS geo-spoofing, and container-level hash breaking.

**NOTE:** This does NOT re-encode your video. Stream copy preserves 100% quality.

### Workflow:
1. Export your final video from CapCut.
2. Upload it to `ShortsMaster/3_CapCut_Exports` on Google Drive.
3. Set `ACTIVE_PRESET` to `MOTIVATION` or `TECH` in Cell 2.
4. Run all cells.
5. Download from `ShortsMaster/4_Final_Upload` and upload to YouTube!
6. Use the printed UPLOAD HELPER to set title, description, tags, and location in YouTube Studio.

In [ ]:
# ============================================================================
# SETUP — Drive Mount & Folder Configuration
# ============================================================================
import os
import subprocess

print("[ ] Checking FFmpeg...")
try:
    subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True, check=True)
    print("[OK] FFmpeg Ready")
except Exception:
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
    print("[OK] FFmpeg Installed")

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

BASE = '/content/gdrive/My Drive/ShortsMaster'
F3_EDIT = f'{BASE}/3_CapCut_Exports'
F4_FINAL = f'{BASE}/4_Final_Upload'

os.makedirs(F3_EDIT, exist_ok=True)
os.makedirs(F4_FINAL, exist_ok=True)

print(f"[OK] Drop CapCut Exports here: {F3_EDIT}")
print(f"[OK] Final Uploads will appear here: {F4_FINAL}")

[ ] Checking FFmpeg...
[OK] FFmpeg Ready
Mounted at /content/gdrive
[OK] Drop CapCut Exports here: /content/gdrive/My Drive/ShortsMaster/3_CapCut_Exports
[OK] Final Uploads will appear here: /content/gdrive/My Drive/ShortsMaster/4_Final_Upload


In [ ]:
# ============================================================================
# PRESETS & CONFIGURATION
# ============================================================================
import random
import hashlib
import time
import struct
from datetime import datetime, timedelta
from pathlib import Path

try:
    import pytz
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'pytz'], check=True)
    import pytz

# CHANGE THIS VARIABLE BEFORE RUNNING!
# Options: "MOTIVATION" or "TECH"
ACTIVE_PRESET = "TECH"

# ============================================================
# MASTER PRESETS — Niche DNA + Geo-Hubs + Device Fingerprint
# ============================================================
PRESETS = {
    "MOTIVATION": {
        "niche": "Motivation & Mindset",
        "dna": (
            "motivation, inspiration, success, mindset, hard-hitting, wholesome, talent, "
            "discipline, grind, stoicism, sigma rule, alpha mentality, hustle, wealth creation, "
            "life lessons, psychology, winner mindset, self improvement, daily motivation, "
            "gym motivation, focus, ambition, never give up, mental toughness, personal growth, "
            "billionaire mindset, deep thoughts, reality check, life advice, motivational speech"
        ),
        "geo": {
            "New York, US": {"coords": "+40.7081-073.9571+012.000/", "tz": "America/New_York"},
            "Miami, US": {"coords": "+25.7617-080.1918+002.000/", "tz": "America/New_York"},
            "Los Angeles, US": {"coords": "+34.0195-118.4912+005.000/", "tz": "America/Los_Angeles"},
            "London, UK": {"coords": "+51.5265-000.0782+010.000/", "tz": "Europe/London"},
            "Toronto, CA": {"coords": "+43.6466-079.4037+008.000/", "tz": "America/Toronto"},
        },
        "upload_description_tpl": (
            "The journey to greatness starts with a single step. "
            "These powerful lessons will transform your mindset and push you to win. "
            "#Shorts #{tag1} #{tag2} #{tag3} #{tag4} #{tag5}"
        ),
    },
    "TECH": {
        "niche": "Tech, Gadgets & Future",
        "dna": (
            "tech, gadgets, future technology, artificial intelligence, AI, innovation, "
            "engineering, robotics, space exploration, smart home, cybersecurity, tech review, "
            "new gadgets, mind-blowing tech, futuristic, apple, tesla, coding, programming, "
            "software, hardware, tech news, daily tech, science, quantum computing, neuralink, "
            "openai, tech tips, amazing inventions, tech hacks, electronics"
        ),
        "geo": {
            "San Francisco, US": {"coords": "+37.7749-122.4194+015.000/", "tz": "America/Los_Angeles"},
            "Austin, US": {"coords": "+30.2672-097.7431+150.000/", "tz": "America/Chicago"},
            "Seattle, US": {"coords": "+47.6062-122.3321+050.000/", "tz": "America/Los_Angeles"},
            "London, UK": {"coords": "+51.5265-000.0782+010.000/", "tz": "Europe/London"},
            "Toronto, CA": {"coords": "+43.6466-079.4037+008.000/", "tz": "America/Toronto"},
        },
        "upload_description_tpl": (
            "The future of technology is here. From AI to robotics, "
            "these mind-blowing innovations will change how you see the world. "
            "#Shorts #{tag1} #{tag2} #{tag3} #{tag4} #{tag5}"
        ),
    },
}

DEVICE = {
    "make": "Samsung",
    "model": "SM-S928B",
    "software": "Android 14",
    "lens_model": "Samsung back quad camera 6.80mm f/1.80",
    "focal_length": "6.80",
    "exposure_time": "1/120",
    "iso_speed": "50",
    "camera_id": "OIS_HARDWARE_ACTIVE",
    "encoder": "MediaCodec",
}

TOP_HASHTAGS = {
    "MOTIVATION": ["Mindset", "Grind", "Success", "Discipline", "Hustle"],
    "TECH": ["AI", "Gadgets", "Innovation", "Future", "Tech"],
}

print(f"[OK] Presets loaded: {list(PRESETS.keys())}")
print(f"[OK] Active preset: {ACTIVE_PRESET}")
print(f"[OK] Device spoof: {DEVICE['model']} ({DEVICE['software']})")

[OK] Presets loaded: ['MOTIVATION', 'TECH']
[OK] Active preset: TECH
[OK] Device spoof: SM-S928B (Android 14)


In [ ]:
# ============================================================================
# FFMPEG METADATA INJECTION — ilst + 3GPP atoms (stream copy, zero quality loss)
# ============================================================================

def build_ffmpeg_metadata_cmd(input_path, output_path, preset, device, city, geo_data):
    timezone = pytz.timezone(geo_data["tz"])
    fake_creation = datetime.now(timezone) - timedelta(
        days=random.randint(1, 3), hours=random.randint(1, 12)
    )
    iso_date = fake_creation.strftime("%Y-%m-%dT%H:%M:%S%z")
    iso_date = iso_date[:-2] + iso_date[-2:]

    fake_mod = fake_creation + timedelta(minutes=random.randint(5, 59))
    iso_mod = fake_mod.strftime("%Y-%m-%dT%H:%M:%S%z")
    iso_mod = iso_mod[:-2] + iso_mod[-2:]

    niche = preset["niche"]
    dna = preset["dna"]

    cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'info',
        '-i', str(input_path),
        '-map_metadata', '-1',

        # --- iTunes ilst atoms (device fingerprint) ---
        '-metadata', f'com.apple.quicktime.make={device["make"]}',
        '-metadata', f'com.apple.quicktime.model={device["model"]}',
        '-metadata', f'com.apple.quicktime.software={device["software"]}',
        '-metadata', f'com.apple.quicktime.camera.focal_length={device["focal_length"]}',
        '-metadata', f'com.apple.quicktime.camera.exposure_time={device["exposure_time"]}',
        '-metadata', f'com.apple.quicktime.camera.iso_speed={device["iso_speed"]}',
        '-metadata', f'com.apple.quicktime.camera.lens_model={device["lens_model"]}',
        '-metadata', f'com.apple.quicktime.camera.identifier={device["camera_id"]}',
        '-metadata', f'encoder={device["encoder"]}',

        # --- 3GPP udta atoms (GPS + timestamps) ---
        '-metadata', f'com.apple.quicktime.creationdate={iso_date}',
        '-metadata', f'com.apple.quicktime.modificationdate={iso_mod}',
        '-metadata', f'location={geo_data["coords"]}',
        '-metadata', f'com.apple.quicktime.location.ISO6709={geo_data["coords"]}',
        '-metadata', 'com.apple.quicktime.location.accuracy.horizontal=4.5',
        '-metadata', f'com.apple.quicktime.timezone={geo_data["tz"]}',

        # --- Niche DNA ---
        '-metadata', 'language=eng',
        '-metadata', f'title={niche} Short',
        '-metadata', f'comment={dna}',
        '-metadata', f'description={dna}',
        '-metadata', f'synopsis={niche} viral content',

        # --- Stream copy (zero quality loss) ---
        '-c', 'copy',
        '-movflags', '+faststart',
        str(output_path),
    ]
    return cmd


def inject_ffmpeg_metadata(input_path, output_path, preset, device, city, geo_data):
    cmd = build_ffmpeg_metadata_cmd(input_path, output_path, preset, device, city, geo_data)
    print(f"  [ffmpeg] Injecting ilst + 3GPP atoms...")
    start = time.time()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        elapsed = time.time() - start
        if result.returncode != 0:
            print(f"  [ffmpeg] FAILED (rc={result.returncode})")
            print(f"  [ffmpeg] stderr: {result.stderr[-500:]}")
            return False
        print(f"  [ffmpeg] Done in {elapsed:.1f}s")
        return True
    except subprocess.TimeoutExpired:
        print(f"  [ffmpeg] TIMEOUT after 120s")
        return False
    except Exception as e:
        print(f"  [ffmpeg] ERROR: {e}")
        return False


print("[OK] FFmpeg metadata injection loaded")

[OK] FFmpeg metadata injection loaded


In [ ]:
# ============================================================================
# XMP INJECTION — Appends UUID atom with XMP XML to MP4 file
# ============================================================================

def build_xmp_xml(niche, dna, device):
    keywords = [k.strip() for k in dna.split(",") if k.strip()]
    keyword_items = "\n".join(f"          <rdf:li>{k}</rdf:li>" for k in keywords)

    xml = f'''<?xpacket begin="" id="W5M0MpCehiHzreSzNTczkc9d"?>
<x:xmpmeta xmlns:x="adobe:ns:meta/">
  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
    <rdf:Description rdf:about=""
      xmlns:dc="http://purl.org/dc/elements/1.1/"
      xmlns:xmp="http://ns.adobe.com/xap/1.0/">
      <dc:subject>
        <rdf:Bag>
{keyword_items}
        </rdf:Bag>
      </dc:subject>
      <dc:description>
        <rdf:Alt>
          <rdf:li xml:lang="en">{niche} viral content</rdf:li>
        </rdf:Alt>
      </dc:description>
      <xmp:CreatorTool>{device["model"]}</xmp:CreatorTool>
    </rdf:Description>
  </rdf:RDF>
</x:xmpmeta>
<?xpacket end="w"?>'''
    return xml.encode('utf-8')


def inject_xmp_atom(file_path, niche, dna, device):
    xmp_uuid = bytes.fromhex("BE7ACFCB97A942E89C71999491E3AFAC")
    xmp_payload = build_xmp_xml(niche, dna, device)

    atom_size = 8 + 16 + len(xmp_payload)
    atom_header = struct.pack('>I', atom_size) + b'uuid'
    atom = atom_header + xmp_uuid + xmp_payload

    print(f"  [xmp] Injecting XMP UUID atom ({len(xmp_payload)} bytes payload)...")
    try:
        with open(file_path, 'ab') as f:
            f.write(atom)
        print(f"  [xmp] Done — appended {len(atom)} bytes")
        return True
    except Exception as e:
        print(f"  [xmp] FAILED: {e}")
        return False


print("[OK] XMP injection loaded")

[OK] XMP injection loaded


In [ ]:
# ============================================================================
# CONTAINER HASH BREAKING — Structural MP4 modifications (no re-encoding)
# All pointer-affecting operations update stco/chunk offsets to prevent
# video corruption.
# ============================================================================

def find_atoms(data, target_type, offset=0, end=None):
    """Find all top-level atoms of a given type in MP4 binary data."""
    if end is None:
        end = len(data)
    atoms = []
    pos = offset
    while pos < end - 8:
        size = struct.unpack('>I', data[pos:pos+4])[0]
        atype = data[pos+4:pos+8]
        if size < 8:
            break
        if atype == target_type:
            atoms.append((pos, size))
        pos += size
    return atoms


def find_stco_in_moov(data, moov_start, moov_end):
    """Find the stco atom inside the moov atom tree.

    Walks the moov subtree looking for 'stbl' -> 'stco' atoms.
    Returns (stco_offset, stco_size) or None.
    """
    pos = moov_start + 8  # skip moov header
    while pos < moov_end - 8:
        size = struct.unpack('>I', data[pos:pos+4])[0]
        atype = data[pos+4:pos+8]
        if size < 8:
            break
        if atype == b'stco':
            return (pos, size)
        # Recurse into container atoms (moov, trak, mdia, minf, stbl)
        if atype in (b'moov', b'trak', b'mdia', b'minf', b'stbl', b'edts'):
            result = find_stco_in_moov(data, pos, pos + size)
            if result:
                return result
        pos += size
    return None


def update_stco_offsets(data, stco_offset, shift_amount):
    """Update all chunk offsets in the stco atom by shift_amount bytes.

    stco format: [4-byte size][4-byte 'stco'][4-byte version+flags][4-byte entry_count][N x 4-byte chunk_offset]
    """
    # Parse stco header
    stco_size = struct.unpack('>I', data[stco_offset:stco_offset+4])[0]
    entry_count_offset = stco_offset + 12  # 4(size) + 4(type) + 4(version+flags)
    entry_count = struct.unpack('>I', data[entry_count_offset:entry_count_offset+4])[0]

    # Convert to mutable bytearray
    data = bytearray(data)

    # Update each chunk offset entry
    for i in range(entry_count):
        entry_offset = entry_count_offset + 4 + (i * 4)
        old_offset = struct.unpack('>I', data[entry_offset:entry_offset+4])[0]
        new_offset = old_offset + shift_amount
        struct.pack_into('>I', data, entry_offset, new_offset)

    return bytes(data)


def add_mdat_padding(file_path):
    """Insert a 'free' atom (padding) before mdat.

    CRITICAL: After inserting padding, all stco chunk offsets must be
    shifted by the padding size, otherwise the video is corrupted.
    """
    pad_size = random.randint(64, 256)
    padding = struct.pack('>I', pad_size) + b'free' + os.urandom(pad_size - 8)

    with open(file_path, 'rb') as f:
        data = f.read()

    # Find mdat atom
    mdat_atoms = find_atoms(data, b'mdat')
    if not mdat_atoms:
        print("  [hash] No mdat found, skipping mdat padding")
        return False

    mdat_offset = mdat_atoms[0][0]

    # Find moov atom to locate stco
    moov_atoms = find_atoms(data, b'moov')
    if not moov_atoms:
        print("  [hash] No moov found, skipping mdat padding (stco unsafe)")
        return False

    moov_start, moov_size = moov_atoms[0]
    stco_result = find_stco_in_moov(data, moov_start, moov_start + moov_size)
    if not stco_result:
        print("  [hash] No stco found, skipping mdat padding (stco unsafe)")
        return False

    stco_offset, stco_size = stco_result

    # Insert padding before mdat
    new_data = data[:mdat_offset] + padding + data[mdat_offset:]

    # Recalculate stco offsets: the stco atom itself may have moved
    # if moov was before mdat. Find where stco ended up.
    moov_shift = pad_size if moov_start < mdat_offset else 0
    new_stco_offset = stco_offset + moov_shift

    # Update all chunk offsets in stco by pad_size
    new_data = update_stco_offsets(new_data, new_stco_offset, pad_size)

    with open(file_path, 'wb') as f:
        f.write(new_data)

    print(f"  [hash] Added {pad_size}-byte free atom before mdat (stco patched)")
    return True


def inject_uuid_atoms(file_path):
    """Append 1-3 random UUID atoms with junk payload.

    Safe: UUID atoms at the end of the file don't affect stco pointers
    because mdat data positions don't change.
    """
    num_uuids = random.randint(1, 3)
    atoms = b''

    for _ in range(num_uuids):
        uuid_bytes = os.urandom(16)
        junk_size = random.randint(32, 128)
        junk = os.urandom(junk_size)
        atom_size = 8 + 16 + junk_size
        atom = struct.pack('>I', atom_size) + b'uuid' + uuid_bytes + junk
        atoms += atom

    with open(file_path, 'ab') as f:
        f.write(atoms)

    print(f"  [hash] Injected {num_uuids} UUID atoms ({len(atoms)} bytes total)")
    return True


def fuzz_creation_time(file_path):
    """Randomize creation/modification timestamps.

    Safe: timestamp values don't affect byte positions or stco pointers.
    """
    with open(file_path, 'rb') as f:
        data = f.read()

    modified = False
    for tag in [b'com.apple.quicktime.creationdate', b'com.apple.quicktime.modificationdate']:
        idx = data.find(tag)
        if idx == -1:
            continue

        search_start = idx + len(tag)
        date_start = data.find(b'20', search_start, search_start + 200)
        if date_start == -1 or date_start > search_start + 100:
            continue

        tz = pytz.timezone(random.choice([
            'America/New_York', 'America/Los_Angeles',
            'America/Chicago', 'Europe/London', 'America/Toronto'
        ]))
        fake = datetime.now(tz) - timedelta(days=random.randint(1, 5), hours=random.randint(0, 23))
        new_date = fake.strftime("%Y-%m-%dT%H:%M:%S%z").encode('ascii')
        new_date = new_date[:len(data[date_start:date_start+25])]

        data = data[:date_start] + new_date + data[date_start + len(new_date):]
        modified = True

    with open(file_path, 'wb') as f:
        f.write(data)

    if modified:
        print(f"  [hash] Fuzzed creation/modification timestamps")
    return modified


def break_container_hash(file_path):
    """Apply 2-3 random structural modifications to change file hash.

    Only uses stco-safe operations. mdat padding updates stco pointers.
    UUID injection and timestamp fuzzing don't affect byte positions.
    """
    techniques = [
        ("mdat padding", add_mdat_padding),
        ("UUID injection", inject_uuid_atoms),
        ("creation time fuzz", fuzz_creation_time),
    ]
    random.shuffle(techniques)

    num_apply = random.randint(2, min(3, len(techniques)))
    applied = 0

    print(f"  [hash] Applying {num_apply} structural modifications...")
    for name, func in techniques[:num_apply]:
        try:
            if func(file_path):
                applied += 1
        except Exception as e:
            print(f"  [hash] {name} failed: {e}")

    print(f"  [hash] Applied {applied}/{num_apply} modifications")
    return applied > 0


print("[OK] Container hash breaking loaded (stco-safe)")

[OK] Container hash breaking loaded (stco-safe)


In [ ]:
# ============================================================================
# UPLOAD HELPER — Generates title, description, tags for YouTube upload
# ============================================================================

def generate_upload_helper(preset, city, device):
    niche = preset["niche"]
    dna = preset["dna"]
    keywords = [k.strip() for k in dna.split(",") if k.strip()]
    tags = TOP_HASHTAGS[ACTIVE_PRESET]

    title = f"{niche} #{tags[0]} #{tags[1]} #{tags[2]}"
    if len(title) > 100:
        title = title[:97] + "..."

    desc_tags = tags[:5]
    description = preset["upload_description_tpl"].format(
        tag1=desc_tags[0], tag2=desc_tags[1], tag3=desc_tags[2],
        tag4=desc_tags[3], tag5=desc_tags[4]
    )

    tag_list = keywords[:18] + [niche, city.split(",")[0]]
    tags_str = ", ".join(tag_list)

    print("\n" + "=" * 60)
    print(f"YOUTUBE UPLOAD HELPER — {ACTIVE_PRESET} PRESET")
    print("=" * 60)
    print(f"\nTITLE:\n{title}")
    print(f"\nDESCRIPTION:\n{description}")
    print(f"\nTAGS:\n{tags_str}")
    print(f"\nLOCATION: Set to \"{city.split(',')[0]}\" in YouTube Studio")
    print(f"\nDEVICE: {device['model']} ({device['software']})")
    print("=" * 60 + "\n")

    return {"title": title, "description": description, "tags": tags_str, "location": city}


print("[OK] Upload helper loaded")

[OK] Upload helper loaded


In [ ]:
# ============================================================================
# RUN PIPELINE — Full Stage 2 metadata injection + hash breaking
# ============================================================================

def find_latest_video(folder):
    exts = {'.mp4', '.mov', '.mkv', '.webm'}
    p = Path(folder)
    vids = [f for f in p.iterdir() if f.suffix.lower() in exts and f.is_file()]
    if not vids:
        return None
    return max(vids, key=lambda f: f.stat().st_mtime)


def run_pipeline():
    if ACTIVE_PRESET not in PRESETS:
        print(f"[ERR] Invalid ACTIVE_PRESET: '{ACTIVE_PRESET}'. Use 'MOTIVATION' or 'TECH'.")
        return

    preset = PRESETS[ACTIVE_PRESET]
    inp_folder = Path(F3_EDIT)
    out_folder = Path(F4_FINAL)
    out_folder.mkdir(parents=True, exist_ok=True)

    latest = find_latest_video(inp_folder)
    if latest is None:
        print("[ERR] No videos found in 3_CapCut_Exports/")
        print(f"      Drop a video in: {F3_EDIT}")
        return

    city, geo_data = random.choice(list(preset["geo"].items()))

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_name = f"VIRAL_{ACTIVE_PRESET}_{latest.stem}_{ts}.mp4"
    ffmpeg_out = out_folder / f"ffmpeg_{out_name}"
    final_out = out_folder / out_name

    in_size = latest.stat().st_size / (1024 * 1024)
    print(f"\n{'=' * 60}")
    print(f"STAGE 2 PIPELINE — {preset['niche'].upper()}")
    print(f"{'=' * 60}")
    print(f"  Input:  {latest.name} ({in_size:.1f} MB)")
    print(f"  Geo:    {city}")
    print(f"  Device: {DEVICE['model']} ({DEVICE['software']})")
    print(f"  Output: {final_out.name}")
    print(f"{'=' * 60}\n")

    with open(latest, 'rb') as f:
        in_hash = hashlib.sha256(f.read()).hexdigest()[:16]
    print(f"[1/5] Input SHA-256: {in_hash}...")

    print(f"\n[2/5] FFmpeg metadata injection...")
    ok = inject_ffmpeg_metadata(latest, ffmpeg_out, preset, DEVICE, city, geo_data)
    if not ok:
        print("[ERR] FFmpeg failed — falling back to raw copy")
        import shutil
        shutil.copy2(latest, ffmpeg_out)

    print(f"\n[3/5] XMP atom injection...")
    inject_xmp_atom(ffmpeg_out, preset["niche"], preset["dna"], DEVICE)

    print(f"\n[4/5] Container hash breaking...")
    break_container_hash(ffmpeg_out)

    print(f"\n[5/5] Validation...")
    ffmpeg_out.rename(final_out)

    if not final_out.exists() or final_out.stat().st_size < 1024:
        print("[ERR] Output file missing or too small")
        return

    try:
        probe = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type',
             '-of', 'csv=p=0', str(final_out)],
            capture_output=True, text=True, timeout=10
        )
        streams = probe.stdout.strip().splitlines()
        has_v = 'video' in streams
        has_a = 'audio' in streams
        if has_v and has_a:
            print("  [OK] Output validated: video + audio streams present")
        elif has_v:
            print("  [!] Output has video only (no audio)")
        else:
            print("  [ERR] Output has no video stream")
            return
    except Exception as e:
        print(f"  [!] Validation skipped: {e}")

    with open(final_out, 'rb') as f:
        out_hash = hashlib.sha256(f.read()).hexdigest()[:16]
    out_size = final_out.stat().st_size / (1024 * 1024)
    hash_changed = in_hash != out_hash

    print(f"\n{'=' * 60}")
    print(f"COMPLETE")
    print(f"{'=' * 60}")
    print(f"  Hash: {in_hash}... -> {out_hash}... ({'CHANGED' if hash_changed else 'SAME'})")
    print(f"  Size: {in_size:.1f} MB -> {out_size:.1f} MB")
    print(f"  File: {final_out.name}")

    generate_upload_helper(preset, city, DEVICE)


run_pipeline()


STAGE 2 PIPELINE — TECH, GADGETS & FUTURE
  Input:  lv_0_20260705112329.mp4 (8.1 MB)
  Geo:    Austin, US
  Device: SM-S928B (Android 14)
  Output: VIRAL_TECH_lv_0_20260705112329_20260705_094103.mp4

[1/5] Input SHA-256: 1f8bf7abd7df7e40...

[2/5] FFmpeg metadata injection...
  [ffmpeg] Injecting ilst + 3GPP atoms...
  [ffmpeg] Done in 0.5s

[3/5] XMP atom injection...
  [xmp] Injecting XMP UUID atom (1832 bytes payload)...
  [xmp] Done — appended 1856 bytes

[4/5] Container hash breaking...
  [hash] Applying 2 structural modifications...
  [hash] mdat padding failed: 'I' format requires 0 <= number <= 4294967295
  [hash] Applied 0/2 modifications

[5/5] Validation...
  [OK] Output validated: video + audio streams present

COMPLETE
  Hash: 1f8bf7abd7df7e40... -> 78e0423fbfd6a8d5... (CHANGED)
  Size: 8.1 MB -> 8.1 MB
  File: VIRAL_TECH_lv_0_20260705112329_20260705_094103.mp4

YOUTUBE UPLOAD HELPER — TECH PRESET

TITLE:
Tech, Gadgets & Future #AI #Gadgets #Innovation

DESCRIPTION:
The f